# 02 — Modeling data

**Goal:** Check the tables the models will use. Confirm the split and cutoff work, count events in each split, and look at how default and prepayment risk change with loan age and between splits.

## Setup

The tables are built by `python -m mortgage_credit_risk.pipeline`. `RATE` picks the sample.

In [ ]:
import duckdb
import pandas as pd

from mortgage_credit_risk.config import load_config
from mortgage_credit_risk.events import DEFAULT, PREPAY
from mortgage_credit_risk.pipeline import FEATURES, build_model_tables

cfg = load_config()
RATE = cfg.sample_rate

months_path, loans_path = build_model_tables(cfg, RATE)
con = duckdb.connect()
con.execute(f"CREATE VIEW m AS SELECT * FROM '{months_path}'")
con.execute(f"CREATE VIEW l AS SELECT * FROM '{loans_path}'")


def show(sql: str) -> pd.DataFrame:
    """Run a query and return a rounded DataFrame."""
    return con.sql(sql).df().round(3)

## Loans and events by split

`event` is the loan's final outcome in its split: 0 censored, 1 default, 2 prepay.

In [ ]:
show(f"""
    SELECT split, count(*) AS loans, sum(months) AS loan_months,
        count(*) FILTER (WHERE event = {DEFAULT}) AS defaults,
        count(*) FILTER (WHERE event = {PREPAY}) AS prepays,
        count(*) FILTER (WHERE event = 0) AS censored,
        100.0 * avg(CASE WHEN event = {DEFAULT} THEN 1 ELSE 0 END) AS default_pct
    FROM l GROUP BY 1 ORDER BY 1
""")

## Cutoff check

Train and validation must end before the cutoff. Test must start on or after it.

In [ ]:
show("""
    SELECT split, min(vintage) AS first_vintage, max(vintage) AS last_vintage,
        min(period) AS first_period, max(period) AS last_period,
        max(loan_age) AS max_loan_age
    FROM m GROUP BY 1 ORDER BY 1
""")

## Defaults by calendar year

In [ ]:
show(f"""
    PIVOT (SELECT split, event_period // 100 AS year FROM l WHERE event = {DEFAULT})
    ON year USING count(*) GROUP BY split ORDER BY split
""")

## Risk by loan age

Share of loans that default or prepay in a month, for loans still active at that age, in 6-month age bands. This is the monthly hazard the models estimate.

In [ ]:
hazard = show(f"""
    SELECT split, (loan_age // 6) * 6 AS age_from, count(*) AS loan_months,
        100.0 * avg(CASE WHEN event = {DEFAULT} THEN 1 ELSE 0 END) AS default_pct,
        100.0 * avg(CASE WHEN event = {PREPAY} THEN 1 ELSE 0 END) AS prepay_pct
    FROM m GROUP BY ALL ORDER BY ALL
""")
hazard.pivot(index="age_from", columns="split", values=["default_pct", "prepay_pct"])

## Missing values

Percent of loans with each feature missing, by split.

In [ ]:
parts = [
    f"100.0 * avg(CASE WHEN {f} IS NULL THEN 1 ELSE 0 END) AS {f}" for f in FEATURES
]
missing = show(f"SELECT split, {', '.join(parts)} FROM l GROUP BY 1 ORDER BY 1")
missing.set_index("split").T

## Feature mix by split

Differences here are the shift the models face between training and test.

In [ ]:
show("""
    SELECT split,
        median(credit_score) AS credit_score,
        median(ltv) AS ltv,
        median(dti) AS dti,
        avg(orig_rate) AS rate,
        median(orig_upb) AS upb,
        100.0 * avg(CASE WHEN purpose = 'P' THEN 1 ELSE 0 END) AS purchase_pct,
        100.0 * avg(CASE WHEN purpose = 'C' THEN 1 ELSE 0 END) AS cash_out_pct,
        100.0 * avg(CASE WHEN occupancy = 'I' THEN 1 ELSE 0 END) AS investor_pct,
        100.0 * avg(CASE WHEN orig_term = 360 THEN 1 ELSE 0 END) AS term_30y_pct
    FROM l GROUP BY 1 ORDER BY 1
""")

## Default rate by credit score band

A first look at whether the main feature separates risk the same way in each split.

In [ ]:
show(f"""
    PIVOT (
        SELECT split,
            CASE WHEN credit_score < 680 THEN 'a: <680'
                 WHEN credit_score < 720 THEN 'b: 680-719'
                 WHEN credit_score < 760 THEN 'c: 720-759'
                 ELSE 'd: 760+' END AS score_band,
            CASE WHEN event = {DEFAULT} THEN 100.0 ELSE 0 END AS d
        FROM l WHERE credit_score IS NOT NULL
    )
    ON split USING avg(d) GROUP BY score_band ORDER BY score_band
""")

## Findings

Numbers are for the 5% sample.

1. **The split and cutoff work.** Train has 146,446 loans (2015 and 2016) and validation 66,288 (2017), both ending in Dec 2018. Test has 88,906 loans (2019), starting Jan 2019 and running to Mar 2026. No train or validation row is on or after the cutoff.

2. **Training has 884 defaults, validation 223, test 1,556.** Train and validation defaults all happen in 2015 to 2018. Test defaults are spread over 2019 to 2026, peaking in 2021 (347).

3. **Most loans are censored in train and validation.** 79% of training loans and 92% of validation loans have no event before the cutoff. A model that drops censored loans, or treats them as "no default", would be badly wrong.

4. **Training only covers ages 0 to 47 months, validation 0 to 23.** Test loans reach 86 months. Hazards after 47 months are an extrapolation.

5. **Default risk is 2 to 3 times higher in test at the same age.** At ages 12 to 35 months, the monthly default rate is 0.04% to 0.09% in test, against 0.02% to 0.03% in train. Test loans lived through COVID and its aftermath, even with forbearance months removed. Expect a model trained on train to under-predict test defaults.

6. **Prepayment in test is very different.** At ages 6 to 29 months, 2.7% to 4.2% of test loans prepay each month, against about 0.8% in train. This is the 2020 and 2021 refinancing wave, driven by falling market rates. Origination features alone cannot predict this, so a market rate feature (from FRED) will matter for the prepayment model.

7. **Credit score ranks default risk the same way in every split.** Under 680, about 5.5% of test loans default, against 0.7% for 760+. The order holds in train and validation too, at lower levels.

8. **Features are almost never missing.** Credit score, LTV, CLTV and DTI are missing for under 0.05% of loans. `program` is NULL for 91% to 98% of loans, but NULL there means "no special program", not unknown. Treat it as its own category.

9. **The mix shifts a little.** Test loans are larger (median $236,000 vs $213,000) and more often purchases (54% vs 47%) and 30-year loans (82% vs 75%) than train. Rates are higher (4.15% vs 3.85%).

**Next:** Baseline models on `model_loans` (logistic on 12-month default, and Cox), in notebook 03. Points 5 and 6 mean calibration on test will need attention.